# Screenshots (Hidden to Save Space)

![image-2](image-2.png)

![image-3](image-3.png)


![image-4](image-4.png)


![image-5](image-5.png)

![image-6](image-6.png)


![image-7](image-7.png)


**Denver Airbnb Analysis - Project Notes:**

- Calendar table grain: One record appears to represent one listing on one calendar date.
  
- Potential calendar key: (listing_id, date).
  
- Review table grain: One record appears to represent one individual review.
  
- Potential review key: id.
  
- Potential relationship: reviews.listing_id → listings.id.
  
- Important limitation: Calendar availability cannot automatically be interpreted as confirmed occupancy.

- id is Airbnb's unique identifier for a listing, while host_id identifies the host.

- neighbourhood_cleansed is calculated by geocoding each listing's latitude/longitude against public/open neighborhood boundaries, making it the better standardized geography field for our neighborhood analysis.

- price is the daily price in local currency, and the $ symbol is an export artifact. That will matter when we clean/cast the field for SQL analysis.

- availability_365 measures days marked available over the next 365 days, but Inside Airbnb explicitly warns that unavailable dates can mean either guest-booked or host-blocked. That confirms why we shouldn't call 365 - availability_365 "occupied nights."

- calculated_host_listings_count is specifically the number of that host's listings found in the current scrape within the city/region, which makes it more useful for our Denver market-concentration analysis than host_total_listings_count, whose calculation Airbnb does not disclose.

- reviews_per_month is an Inside Airbnb calculated field representing average monthly reviews over the listing's lifetime, and the dictionary even documents its approximate calculation.

--------

- There are 10 more distinct listing IDs represented in the calendar than there are listings in detailed_listings.csv.

- Every listing id has exactly 365 rows each in detailed_calendar

- Finding: 10 listing IDs represented in the detailed calendar are absent from the detailed listings snapshot.

- Referential integrity: The detailed reviews table contains 95 historical reviews associated with 8 listing IDs that are not present in the current listings snapshot. Additionally, 638 current listings have no records in the detailed reviews table.

- Issue: price is stored as text and contains currency symbols.
Treatment: Remove the currency formatting and convert the field to a numeric datatype before performing price calculations.

- Price is strongly right-skewed. A global IQR cutoff over-flags listings because room types have different price structures. Room-type-specific IQR is a better screening method, but flagged values should not be removed automatically because many may be legitimate high-end listings.

- Roughly 300 of the 4,251 priced listings are being flagged as statistical upper outliers. That’s about 7% of priced listings, which is plausible for a heavily right-skewed market.

- The biggest finding is the difference between the average ($276.04) and median ($176.45). The mean is much higher, which is strong evidence that the price distribution is right-skewed: a relatively small number of very expensive listings are pulling the average upward.

- From an analysis perspective, however, it's extremely important. Denver's Airbnb supply in this dataset is overwhelmingly entire homes/apartments: 85.71%. That tells us room_type should probably become one of our major segmentation dimensions throughout the project.

detailed_listings
- Grain: One row per listing
- Primary key candidate: id
- Uniqueness: Confirmed in current dataset
- NULLs in key: None

room_type and property_type
- 100% complete with four standardized categories. Entire home/apartment listings dominate the Denver dataset (85.71%), followed by private rooms (12.78%). No cleaning required. Retain as a key categorical segmentation variable.
- room_type answers "How much of the property does the guest get?", while property_type gives a more specific description of "What kind of accommodation/property is it?"
- "property_type contained 38 categories with substantial class imbalance. Inspection indicated that rare categories represented legitimate accommodation types rather than obvious data-quality errors. The original field was therefore retained, with broader analytical groupings considered for segment-level analysis."
- Entire home/apt - Entire home has the most listings at 1589 listings and 32.17% share. 
- I'm noticing there are private room types that could be better classified in 'Hotel room', including room in hotel, ...aparthotel, ...boutique hotel, ...hostel.
----------
- room_type: Low-cardinality broad classification with four complete, standardized categories. Useful for high-level market segmentation.
- property_type: Higher-cardinality field with 38 legitimate accommodation categories and a long-tailed distribution. Provides additional structural detail within each room type. Retain the raw field; broader derived categories may be created later if needed for analysis.
- room_type and property_type are complementary rather than redundant. room_type provides a broad classification of the rental arrangement, while property_type provides more granular information about the physical accommodation. Therefore, both fields should be retained.

Data-Cleaning for price 
- Issue: price is stored as text and contains currency symbols.
- Treatment: Remove the currency formatting and convert the field to a numeric datatype before performing price calculations. New column: cleaned_price

Listing counts for price thresholds
- $500–$999.99: 336 listings
- $1,000–$1,999.99: 53 listings
- $2,000+: 9 listings

Errors, Extreme Values, and Outliers 
- Data error — a value is demonstrably incorrect or unusable.
- Extreme value — a legitimate but unusually large/small observation.
- Statistical outlier — a value falling outside a defined statistical criterion such as 1.5×IQR.

------------

neighbourhood_cleansed
- no missing values
- 77 legitimate categories
- no apparent capitalization/whitespace duplication
- meaningful variation in listing counts
- strong analytical relevance
- compatible with our later geographic analysis
- neighbourhoods.csv contains 78 Denver neighborhoods. All 77 neighborhoods represented in the current listings snapshot match the reference table. Sun Valley is present in the neighborhood reference data but has no listings in the current detailed_listings.csv snapshot.

------------

accommodates
- accommodates contains no null or zero values and ranges from 1–16 guests. Listings at the upper bound appear consistent with large-group properties based on bedroom, bathroom, and listing-description information.
- There's another important consideration: 16 appears repeatedly. You may eventually discover that 16 is effectively a platform/category ceiling rather than the exact physical maximum occupancy of every property. For this project, however, you don't need to resolve that yet. Just don't interpret 16 as necessarily meaning "this property can accommodate exactly 16 and no more."

----------

bathrooms and bathrooms_text
- bathrooms_text reliably represents bathroom count and provides bathroom information for many listings where the numeric bathrooms field is missing. Among records where both fields are populated, the numeric values are consistent except for one identified discrepancy.
- When both fields are populated, bathrooms and the numeric bathroom count represented by bathrooms_text are highly consistent. One conflicting record was identified (bathrooms = 2, bathrooms_text = '3.5 baths'). This supports using bathrooms_text as a fallback source for missing numeric bathroom counts, subject to explicit handling of special text values.
- bathrooms_text is the more complete bathroom field. It contains usable bathroom information for 885 of the 887 records where numeric bathrooms is missing. Conversely, only 4 records have a numeric bathrooms value while bathrooms_text is missing. When both fields are populated, the values are highly consistent aside from one identified discrepancy.

----------

bedrooms/beds
- bedrooms and beds exhibit partially overlapping missingness. Approximately 75.6% of listings have both fields populated, while only 1.4% are missing both. Because each field is available for hundreds of records where the other is missing, both should be retained rather than treated as substitutes.
- One listing contains an unusually inconsistent capacity profile: 15 bedrooms, 50 beds, but an accommodates value of only 2. Because several capacity fields conflict, the record is flagged as a potential data-quality anomaly rather than automatically corrected or removed.
- For the eventual analysis, I would treat accommodates as your primary capacity variable, with bedrooms and beds as supplementary property characteristics. It's complete and directly represents the listing's stated guest capacity.

---------

minimum_nights
- Minimum-stay requirements are highly concentrated at a small number of values. One-night stays are the most common requirement (27.1%), while 29- and 30-night minimums collectively account for approximately 46.9% of listings. This suggests a substantial segment of the Denver Airbnb market is oriented toward longer stays. Extreme minimum-stay values also exist, reaching 500 nights, but these represent atypical listings and should not be interpreted as representative of the market.
- Minimum-stay policies show a highly polarized distribution. Approximately 46.9% of listings require 29–30 nights, while approximately 49.8% allow stays of six nights or fewer. Listings with minimum stays between 7 and 28 nights are comparatively rare at less than 1%.

----------

availability
- Inside Airbnb's available calendar status doesn't tell us why a date is unavailable. A host may have blocked dates, stopped accepting reservations temporarily, restricted the calendar, or the date may actually be booked.
- Calendar availability is substantially lower in the near term. Listings average approximately 30% availability over the next 30 days, compared with roughly 50–62% across subsequent future periods. This pattern may reflect a combination of reservations, host calendar restrictions, and other availability settings and should not be interpreted directly as occupancy.
- Availability fields were complete and remained within their expected bounds (0–30, 0–60, 0–90, and 0–365). Availability was substantially lower in the immediate 30-day window than in later future periods. Zero availability was particularly common over the next 30 days (35.1%), declining as the observation window expanded. Because unavailable calendar dates may represent bookings or host-imposed restrictions, availability is treated as a calendar-availability measure rather than an occupancy measure.
- The cumulative availability fields are logically consistent across the dataset. No listing has a shorter-window availability value greater than a longer-window availability value.

Data Dictionary: https://docs.google.com/spreadsheets/d/1iWCNJcSutYqpULSQHlNyGInUvHg2BoUGoNRIGa6Szc4/edit?usp=sharing

# Understanding the Schema of the Datasets and Data Profiling/Quality Assessment

In [3]:
-- Explore the data in the table
DESCRIBE 'detailed_calendar.csv';

,column_name,column_type,null,key,default,extra
0,listing_id,BIGINT,YES,None,None,None
1,date,DATE,YES,None,None,None
2,available,BOOLEAN,YES,None,None,None
3,minimum_nights,BIGINT,YES,None,None,None
4,maximum_nights,BIGINT,YES,None,None,None


In [19]:
SELECT 
	COUNT(*) AS total_rows_count,
	COUNT(DISTINCT listing_id) AS listings_count,
	MIN(date) AS earliest_date,
	MAX(date) AS latest_date,
	COUNT(CASE WHEN available = TRUE THEN 1 END) AS available_rows,
	COUNT(CASE WHEN available = FALSE THEN 1 END) AS unavailable_rows
FROM 'detailed_calendar.csv';

,total_rows_count,listings_count,earliest_date,latest_date,available_rows,unavailable_rows
0,1806385,4949,2026-06-30 00:00:00+00:00,2027-07-01 00:00:00+00:00,1021669,784716


In [4]:
DESCRIBE 'detailed_listings.csv';

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,listing_url,VARCHAR,YES,None,None,None
2,scrape_id,BIGINT,YES,None,None,None
3,last_scraped,DATE,YES,None,None,None
4,source,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
85,calculated_host_listings_count,BIGINT,YES,None,None,None
86,calculated_host_listings_count_entire_homes,BIGINT,YES,None,None,None
87,calculated_host_listings_count_private_rooms,BIGINT,YES,None,None,None
88,calculated_host_listings_count_shared_rooms,BIGINT,YES,None,None,None


In [10]:
SELECT 
	COUNT(*) AS total_rows_count,
	COUNT(DISTINCT id) AS id_count,
	COUNT(CASE WHEN id IS NULL THEN 1 END) AS id_null_count,
	COUNT(DISTINCT host_id) AS host_id_count,
	COUNT(CASE WHEN host_id IS NULL THEN 1 END) AS host_id_null_count,
	MIN(first_review) AS earliest_first_review,
	MAX(first_review) AS latest_first_review,
	MIN(last_review) AS earliest_last_review,
	MAX(last_review) AS latest_last_review
FROM 'detailed_listings.csv';

,total_rows_count,id_count,id_null_count,host_id_count,host_id_null_count,earliest_first_review,latest_first_review,earliest_last_review,latest_last_review
0,4939,4939,0,3318,0,2008-08-28 00:00:00+00:00,2026-06-28 00:00:00+00:00,2015-07-23 00:00:00+00:00,2026-06-30 00:00:00+00:00


In [5]:
DESCRIBE 'detailed_reviews.csv';

,column_name,column_type,null,key,default,extra
0,listing_id,BIGINT,YES,None,None,None
1,id,BIGINT,YES,None,None,None
2,date,DATE,YES,None,None,None
3,reviewer_id,BIGINT,YES,None,None,None
4,reviewer_name,VARCHAR,YES,None,None,None
5,comments,VARCHAR,YES,None,None,None


In [8]:
SELECT 
	COUNT(*) AS total_rows_count,
	COUNT(DISTINCT listing_id) AS listings_count,
	COUNT(CASE WHEN listing_id IS NULL THEN 1 END) AS null_listings_count,
	COUNT(DISTINCT id) AS distinct_review_ids, 
	COUNT(CASE WHEN id IS NULL THEN 1 END) AS null_review_ids,
	COUNT(DISTINCT reviewer_id) AS reviewer_count,
	COUNT(CASE WHEN reviewer_id IS NULL THEN 1 END) AS null_reviewer_count,
	MIN(date) AS earliest_review,
	MAX(date) AS latest_review
FROM 'detailed_reviews.csv';

,total_rows_count,listings_count,null_listings_count,distinct_review_ids,null_review_ids,reviewer_count,null_reviewer_count,earliest_review,latest_review
0,354741,4309,0,354741,0,316556,0,2008-08-28 00:00:00+00:00,2026-06-30 00:00:00+00:00


In [6]:
DESCRIBE 'neighbourhoods.csv';

,column_name,column_type,null,key,default,extra
0,neighbourhood_group,VARCHAR,YES,None,None,None
1,neighbourhood,VARCHAR,YES,None,None,None


In [9]:
SELECT 
	COUNT(*) AS total_rows,
	COUNT(DISTINCT neighbourhood_group) AS neighbourhood_group_count,
	COUNT(DISTINCT neighbourhood) AS neighbourhood_count
FROM 'neighbourhoods.csv';

,total_rows,neighbourhood_group_count,neighbourhood_count
0,78,0,78


## Answers to Follow-Up Questions to Confirm Relationship Validation

In [20]:
/* Does every listing have exactly 365 records? */

SELECT 
	listing_id,
	COUNT(*) AS calendar_rows
FROM 'detailed_calendar.csv'
GROUP BY listing_id
HAVING COUNT(*) != 365
ORDER BY listing_id;

,listing_id,calendar_rows


In [12]:
/* Which calendar.listing_ids do not exist in listings.id? */

SELECT 
	DISTINCT c.listing_id,
	l.id
FROM 'detailed_calendar.csv' AS c
LEFT JOIN 'detailed_listings.csv' AS l
	ON c.listing_id = l.id
WHERE l.id IS NULL;

,listing_id,id
0,1663911003117317942,<NA>
1,1541416314294129738,<NA>
2,1663911003765151859,<NA>
3,1045288088562648513,<NA>
4,1703706462421883312,<NA>
5,1541416240915518927,<NA>
6,1613930943834383250,<NA>
7,1703706463365002102,<NA>
8,1541416240576378877,<NA>
9,1350903199350412273,<NA>


In [23]:
/* Which reviews.listing_ids do not exist in listings.id? */ 

SELECT 
	COUNT(DISTINCT r.listing_id) AS missing_listings,
	COUNT(*) AS missing_reviews
FROM 'detailed_reviews.csv' AS r
LEFT JOIN 'detailed_listings.csv' AS l
	ON r.listing_id = l.id
WHERE l.id IS NULL;

,missing_listings,missing_reviews
0,8,95


In [22]:
/* Which listings have no reviews in detailed_reviews? */ 

SELECT 
	l.id,
	r.listing_id
FROM 'detailed_listings.csv' AS l
LEFT JOIN 'detailed_reviews.csv' AS r
	ON l.id = r.listing_id
WHERE r.listing_id IS NULL;

,id,listing_id
0,921034342240660670,<NA>
1,962074234097337408,<NA>
2,968742503739637809,<NA>
3,1011467506862097425,<NA>
4,1072094310086905596,<NA>
...,...,...
633,49841433,<NA>
634,554636571544399637,<NA>
635,719385608083897860,<NA>
636,754625064965668024,<NA>


In [17]:
/* Does every listings.neighbourhood_cleansed match a neighbourhoods.neighbourhood? */

SELECT 
	l.neighbourhood_cleansed,
	n.neighbourhood
FROM 'detailed_listings.csv' AS l
LEFT JOIN 'neighbourhoods.csv' AS n
	ON l.neighbourhood_cleansed = n.neighbourhood
WHERE n.neighbourhood IS NULL;

,neighbourhood_cleansed,neighbourhood


# Profiling the Analytical Variables

## Price

In [29]:
WITH price AS (
	SELECT 
		COUNT(*) AS total_values,
		COUNT(price) AS price_values,
		COUNT(CASE WHEN price IS NULL THEN 1 END) AS null_price_values
	FROM 'detailed_listings.csv'
)

SELECT *,
	ROUND(price_values/total_values * 100, 2) || '%' AS percentage_of_listings_with_price
FROM price;

,total_values,price_values,null_price_values,percentage_of_listings_with_price
0,4939,4251,688,86.07%


In [32]:
/* Modifying the price column to be read as a number and able to be used in calculations */ 

SELECT 
	price,
	CAST (
		REPLACE(
			REPLACE(price, '$', ''),
			',',
			''
		)
		AS DECIMAL(10, 2)
	) AS cleaned_price
FROM 'detailed_listings.csv'
LIMIT 10;

,price,cleaned_price
0,$106.35,106.35
1,$32.35,32.35
2,$80.00,80.00
3,$44.27,44.27
4,$191.72,191.72
5,$94.75,94.75
6,$101.66,101.66
7,$35.47,35.47
8,$102.71,102.71
9,$78.67,78.67


In [26]:
SELECT 
	DISTINCT TYPEOF(price) AS price_data_type
FROM 'detailed_listings.csv';

,price_data_type
0,VARCHAR


In [4]:
/* Calculating min, max, avg, median, zero price values, and over xxx amounts for the cleaned prices */

WITH cleaned_prices AS (
	SELECT 
		CAST (
			REPLACE(
				REPLACE(price, '$', ''),
				',',
				''
			)
			AS DECIMAL(10, 2)
		) AS cleaned_price
	FROM 'detailed_listings.csv'
) 

SELECT 
	MIN(cleaned_price) AS min_price,
	MAX(cleaned_price) AS max_price,
	ROUND(AVG(cleaned_price), 2) AS avg_price,
	MEDIAN(cleaned_price) AS median_price,
	COUNT(CASE WHEN cleaned_price = 0 THEN 1 END) AS zero_price_values,
	COUNT(CASE WHEN cleaned_price > 500 AND cleaned_price < 1000 THEN 1 END) AS over_500_listings,
	COUNT(CASE WHEN cleaned_price > 1000 AND cleaned_price < 2000 THEN 1 END) AS over_1000_listings,
	COUNT(CASE WHEN cleaned_price > 2000 THEN 1 END) AS over_2000_listings
FROM cleaned_prices;

,min_price,max_price,avg_price,median_price,zero_price_values,over_500_listings,over_1000_listings,over_2000_listings
0,7.38,110616.6,276.04,176.45,0,336,53,9


In [4]:
/* Showing the top 20 most expensive listings */ 

WITH cleaned_prices AS (
	SELECT 
		id, 
		name,
		CAST (
			REPLACE(
				REPLACE(price, '$', ''),
				',',
				''
			)
			AS DECIMAL(10, 2)
		) AS cleaned_price,
		property_type, room_type, accommodates, bedrooms, bathrooms, minimum_nights, availability_365, number_of_reviews, neighbourhood_cleansed
	FROM 'detailed_listings.csv'
) 

SELECT *
FROM cleaned_prices
ORDER BY cleaned_price DESC
LIMIT 20;

,id,name,cleaned_price,property_type,room_type,accommodates,bedrooms,bathrooms,minimum_nights,availability_365,number_of_reviews,neighbourhood_cleansed
0,21712656,Marion,110616.60,Entire condo,Entire home/apt,2,<NA>,NaN,30,31,21,Cheesman Park
1,35382615,Newly Redecorated! Central Park Family Favorite!,23573.00,Entire home,Entire home/apt,5,2,2.0,1,91,162,Stapleton
2,36515976,"The Ramble Hotel, Bunkhouse",9690.00,Room in boutique hotel,Private room,2,1,1.0,29,365,6,Five Points
3,687686195085166733,New Urban Oasis - with covered deck and worksp...,5877.50,Entire townhouse,Entire home/apt,12,3,3.0,2,365,45,Chaffee Park
4,45803041,Escape to Denver King stile,3526.60,Private room in rental unit,Private room,2,<NA>,1.0,29,270,1,East Colfax
5,1195056124079372458,The Castle Marne,2995.00,Castle,Entire home/apt,10,5,8.0,1,295,2,City Park West
6,28057388,Huge Denver Townhouse with 4 Bedrooms + Office,2941.50,Entire townhouse,Entire home/apt,9,4,3.5,2,85,132,Skyland
7,655281960656437689,New build house with heated pool!,2373.67,Entire home,Entire home/apt,12,6,6.0,1,319,46,Belcaro
8,583955778204351946,Historic Charm - Downtown Denver Gem,2116.00,Entire home,Entire home/apt,16,5,5.0,2,319,39,Cheesman Park
9,1062497121327188829,Downtown's Largest Rental - Walk to Everything!,1988.62,Entire condo,Entire home/apt,16,8,4.0,1,344,12,Five Points


In [26]:
/* Showing the top 20 cheapest listings */ 

WITH cleaned_prices AS (
	SELECT 
		id, 
		name,
		CAST (
			REPLACE(
				REPLACE(price, '$', ''),
				',',
				''
			)
			AS DECIMAL(10, 2)
		) AS cleaned_price,
		property_type, room_type, accommodates, bedrooms, bathrooms, minimum_nights, availability_365, number_of_reviews, neighbourhood_cleansed, listing_url
	FROM 'detailed_listings.csv'
) 

SELECT *
FROM cleaned_prices
ORDER BY cleaned_price 
LIMIT 20;

,id,name,cleaned_price,property_type,room_type,accommodates,bedrooms,bathrooms,minimum_nights,availability_365,number_of_reviews,neighbourhood_cleansed,listing_url
0,1277722996090475527,Urban Oasis in LoHi Neighborhood,7.38,Entire guesthouse,Entire home/apt,2,1,NaN,365,365,0,Highland,https://www.airbnb.com/rooms/1277722996090475527
1,10401469,Large Bedrm Gym Views PvtBath Prkg,9.66,Private room in rental unit,Private room,2,<NA>,NaN,180,180,43,Capitol Hill,https://www.airbnb.com/rooms/10401469
2,18506974,1b/1b Downtown Denver condo - RiNo art district,13.17,Entire condo,Entire home/apt,2,1,NaN,180,364,1,Five Points,https://www.airbnb.com/rooms/18506974
3,48981489,Bedroom w/ private bathroom by Cheesman Park,14.82,Private room in condo,Private room,1,2,NaN,100,163,1,Cheesman Park,https://www.airbnb.com/rooms/48981489
4,43686804,Cap Hill Penthouse Condo w/ Parking & Views,14.99,Entire condo,Entire home/apt,2,1,NaN,140,273,5,Capitol Hill,https://www.airbnb.com/rooms/43686804
5,1693034151271529508,The Denver Dorms,15.20,Shared room in home,Shared room,1,1,NaN,29,336,0,Overland,https://www.airbnb.com/rooms/1693034151271529508
6,1693031392354183041,The Denver Dorms,15.68,Shared room in home,Shared room,1,1,1.0,29,340,0,Overland,https://www.airbnb.com/rooms/1693031392354183041
7,1693030303095498132,The Denver Dorms,15.73,Shared room in home,Shared room,1,1,1.0,29,319,0,Overland,https://www.airbnb.com/rooms/1693030303095498132
8,1693019533486608695,The Denver Dorms,15.77,Shared room in home,Shared room,1,1,1.0,29,334,0,Overland,https://www.airbnb.com/rooms/1693019533486608695
9,1693036036633322447,The Denver Dorms,15.80,Shared room in home,Shared room,1,1,1.0,29,337,0,Overland,https://www.airbnb.com/rooms/1693036036633322447


In [8]:
/* Calculating quartiles/bounds and showing the listings above the upper bound */ 

WITH cleaned_prices AS (
	SELECT 
		id, property_type, room_type, accommodates, neighbourhood_cleansed,
		CAST (
			REPLACE(
				REPLACE(price, '$', ''),
				',',
				''
			)
			AS DECIMAL(10, 2)
		) AS cleaned_price,
	FROM 'detailed_listings.csv'
),

cleaned_prices_quartiles AS (
	SELECT 
		QUANTILE_CONT(cleaned_price, 0.25) AS q1,
		MEDIAN(cleaned_price) AS median_price,
		QUANTILE_CONT(cleaned_price, 0.75) AS q3
	FROM cleaned_prices
),

cleaned_prices_iqr AS (
	SELECT 
		q1, median_price, q3, q3 - q1 AS iqr
	FROM cleaned_prices_quartiles
),

price_bounds AS (
	SELECT 
		q1, median_price, q3, iqr,
		q1 - (1.5 * iqr) AS lower_bound,
		q3 + (1.5 * iqr) AS upper_bound
	FROM cleaned_prices_iqr
)

SELECT *
FROM cleaned_prices AS p
CROSS JOIN price_bounds AS b
WHERE cleaned_price > upper_bound
ORDER BY cleaned_price;

,id,property_type,room_type,accommodates,neighbourhood_cleansed,cleaned_price,q1,median_price,q3,iqr,lower_bound,upper_bound
0,761097400434841164,Entire home,Entire home/apt,6,Highland,569.07,105.99,176.45,291.0,185.01,-171.525,568.515
1,1460296057594263746,Entire home,Entire home/apt,10,Highland,569.98,105.99,176.45,291.0,185.01,-171.525,568.515
2,1543768181548463624,Entire rental unit,Entire home/apt,6,Union Station,570.00,105.99,176.45,291.0,185.01,-171.525,568.515
3,1140660126353780965,Entire home,Entire home/apt,10,Stapleton,570.67,105.99,176.45,291.0,185.01,-171.525,568.515
4,1594367780968298088,Private room in home,Private room,2,Hampden,570.71,105.99,176.45,291.0,185.01,-171.525,568.515
...,...,...,...,...,...,...,...,...,...,...,...,...
295,45803041,Private room in rental unit,Private room,2,East Colfax,3526.60,105.99,176.45,291.0,185.01,-171.525,568.515
296,687686195085166733,Entire townhouse,Entire home/apt,12,Chaffee Park,5877.50,105.99,176.45,291.0,185.01,-171.525,568.515
297,36515976,Room in boutique hotel,Private room,2,Five Points,9690.00,105.99,176.45,291.0,185.01,-171.525,568.515
298,35382615,Entire home,Entire home/apt,5,Stapleton,23573.00,105.99,176.45,291.0,185.01,-171.525,568.515


In [9]:
/* Showing the count of room types above the upper bound */ 

WITH cleaned_prices AS (
	SELECT 
		id, property_type, room_type, accommodates, neighbourhood_cleansed,
		CAST (
			REPLACE(
				REPLACE(price, '$', ''),
				',',
				''
			)
			AS DECIMAL(10, 2)
		) AS cleaned_price,
	FROM 'detailed_listings.csv'
),

cleaned_prices_quartiles AS (
	SELECT 
		QUANTILE_CONT(cleaned_price, 0.25) AS q1,
		MEDIAN(cleaned_price) AS median_price,
		QUANTILE_CONT(cleaned_price, 0.75) AS q3,
		QUANTILE_CONT(cleaned_price, 0.75) - QUANTILE_CONT(cleaned_price, 0.25) AS iqr
	FROM cleaned_prices
),

price_bounds AS (
	SELECT 
		q1, median_price, q3, iqr,
		q1 - (1.5 * iqr) AS lower_bound,
		q3 + (1.5 * iqr) AS upper_bound
	FROM cleaned_prices_quartiles
),

cleaned_prices_with_bounds AS (
	SELECT *
	FROM cleaned_prices AS p
	CROSS JOIN price_bounds AS b
)

SELECT 
	room_type,
	COUNT(*) AS outlier_count
FROM cleaned_prices_with_bounds
WHERE cleaned_price > upper_bound
GROUP BY room_type
ORDER BY outlier_count DESC;

,room_type,outlier_count
0,Entire home/apt,285
1,Private room,13
2,Hotel room,2


In [18]:
/* Showing the count of room types above their respective upper bounds */ 

WITH cleaned_prices AS (
	SELECT 
		id, property_type, room_type, accommodates, neighbourhood_cleansed,
		CAST (
			REPLACE(
				REPLACE(price, '$', ''),
				',',
				''
			)
			AS DECIMAL(10, 2)
		) AS cleaned_price,
	FROM 'detailed_listings.csv'
),

cleaned_prices_quartiles AS (
	SELECT 
		room_type,
		COUNT(cleaned_price) AS listing_count,
		QUANTILE_CONT(cleaned_price, 0.25) AS q1,
		MEDIAN(cleaned_price) AS median_price,
		QUANTILE_CONT(cleaned_price, 0.75) AS q3,
		QUANTILE_CONT(cleaned_price, 0.75) - QUANTILE_CONT(cleaned_price, 0.25) AS iqr
	FROM cleaned_prices
	GROUP BY room_type
),

price_bounds AS (
	SELECT 
		room_type, listing_count, q1, median_price, q3, iqr,
		q1 - (1.5 * iqr) AS lower_bound,
		q3 + (1.5 * iqr) AS upper_bound
	FROM cleaned_prices_quartiles
)

SELECT 
	p.room_type,
	COUNT(*) AS outlier_count
FROM cleaned_prices AS P
INNER JOIN price_bounds AS b
	ON p.room_type = b.room_type
WHERE p.cleaned_price > b.upper_bound
GROUP BY p.room_type
ORDER BY outlier_count DESC;

,room_type,outlier_count
0,Entire home/apt,213
1,Private room,22


In [25]:
/* Inspecting the most extreme room-type-specific outliers and decide whether any are clearly implausible */ 

WITH cleaned_prices AS (
	SELECT 
		id, name, property_type, room_type, bedrooms, bathrooms, minimum_nights, number_of_reviews, review_scores_rating, availability_365, listing_url, accommodates, neighbourhood_cleansed,
		CAST (
			REPLACE(
				REPLACE(price, '$', ''),
				',',
				''
			)
			AS DECIMAL(10, 2)
		) AS cleaned_price,
	FROM 'detailed_listings.csv'
),

cleaned_prices_quartiles AS (
	SELECT 
		room_type,
		COUNT(cleaned_price) AS listing_count,
		QUANTILE_CONT(cleaned_price, 0.25) AS q1,
		MEDIAN(cleaned_price) AS median_price,
		QUANTILE_CONT(cleaned_price, 0.75) AS q3,
		QUANTILE_CONT(cleaned_price, 0.75) - QUANTILE_CONT(cleaned_price, 0.25) AS iqr
	FROM cleaned_prices
	GROUP BY room_type
),

price_bounds AS (
	SELECT 
		room_type, listing_count, q1, median_price, q3, iqr,
		q1 - (1.5 * iqr) AS lower_bound,
		q3 + (1.5 * iqr) AS upper_bound
	FROM cleaned_prices_quartiles
)

SELECT 
	p.id, p.name, p.property_type, p.room_type, p.cleaned_price, p.bedrooms, p.bathrooms, p.minimum_nights, p.number_of_reviews, p.review_scores_rating, p.availability_365, p.listing_url, p.accommodates, p.neighbourhood_cleansed, 
	CASE WHEN p.cleaned_price > b.upper_bound THEN 1 ELSE 0 END AS price_outlier_flag,
	b.listing_count, b.q1, b.median_price, b.q3, b.iqr, b.lower_bound, b.upper_bound
FROM cleaned_prices AS p
INNER JOIN price_bounds AS b
	ON p.room_type = b.room_type 
WHERE p.cleaned_price > b.upper_bound
ORDER BY p.cleaned_price DESC;

,id,name,property_type,room_type,cleaned_price,bedrooms,bathrooms,minimum_nights,number_of_reviews,review_scores_rating,availability_365,listing_url,accommodates,neighbourhood_cleansed,price_outlier_flag,listing_count,q1,median_price,q3,iqr,lower_bound,upper_bound
0,21712656,Marion,Entire condo,Entire home/apt,110616.60,<NA>,NaN,30,21,4.71,31,https://www.airbnb.com/rooms/21712656,2,Cheesman Park,1,3629,115.80,190.59,323.30,207.50,-195.450,634.550
1,35382615,Newly Redecorated! Central Park Family Favorite!,Entire home,Entire home/apt,23573.00,2,2.0,1,162,5.00,91,https://www.airbnb.com/rooms/35382615,5,Stapleton,1,3629,115.80,190.59,323.30,207.50,-195.450,634.550
2,36515976,"The Ramble Hotel, Bunkhouse",Room in boutique hotel,Private room,9690.00,1,1.0,29,6,4.67,365,https://www.airbnb.com/rooms/36515976,2,Five Points,1,554,56.74,106.00,162.63,105.89,-102.095,321.465
3,687686195085166733,New Urban Oasis - with covered deck and worksp...,Entire townhouse,Entire home/apt,5877.50,3,3.0,2,45,5.00,365,https://www.airbnb.com/rooms/687686195085166733,12,Chaffee Park,1,3629,115.80,190.59,323.30,207.50,-195.450,634.550
4,45803041,Escape to Denver King stile,Private room in rental unit,Private room,3526.60,<NA>,1.0,29,1,5.00,270,https://www.airbnb.com/rooms/45803041,2,East Colfax,1,554,56.74,106.00,162.63,105.89,-102.095,321.465
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230,23072906,DTC/ I 225 /I 25 bedroom 30 day discount,Private room in home,Private room,390.54,<NA>,1.0,29,6,4.83,365,https://www.airbnb.com/rooms/23072906,1,Hampden South,1,554,56.74,106.00,162.63,105.89,-102.095,321.465
231,1567628402204033027,Rated #1 hotel in Denver by Travel + Leisure,Room in hotel,Private room,362.00,1,NaN,1,0,NaN,335,https://www.airbnb.com/rooms/1567628402204033027,2,Union Station,1,554,56.74,106.00,162.63,105.89,-102.095,321.465
232,1567628406535049292,Chic hotel room with two queen beds,Room in hotel,Private room,362.00,1,1.0,1,1,5.00,349,https://www.airbnb.com/rooms/1567628406535049292,2,Union Station,1,554,56.74,106.00,162.63,105.89,-102.095,321.465
233,53970186,Eclectic 1-bedroom studio aparthotel,Room in hotel,Private room,330.00,1,1.5,1,5,5.00,362,https://www.airbnb.com/rooms/53970186,1,Cole,1,554,56.74,106.00,162.63,105.89,-102.095,321.465


## room_type

In [29]:
/* Listing counts for each room type and how common each category is */ 

WITH room_type_listings AS (
	SELECT 
		room_type,
		COUNT(*) AS listings
	FROM 'detailed_listings.csv'
	GROUP BY room_type
	ORDER BY listings DESC
),

total AS (
	SELECT 
		COUNT(*) AS total_listings
	FROM 'detailed_listings.csv' 
)

SELECT 
	r.room_type, r.listings,
	ROUND(r.listings/t.total_listings * 100, 2) AS percent_of_total
FROM room_type_listings AS r
CROSS JOIN total AS t
ORDER BY percent_of_total DESC;

,room_type,listings,percent_of_total
0,Entire home/apt,4233,85.71
1,Private room,631,12.78
2,Shared room,52,1.05
3,Hotel room,23,0.47


In [41]:
/* Checking for null values in room types */ 

SELECT 
	room_type,
	COUNT(CASE WHEN room_type IS NULL THEN 1 END) AS null_values
FROM 'detailed_listings.csv'
GROUP BY room_type
ORDER BY null_values DESC;

,room_type,null_values
0,Shared room,0
1,Entire home/apt,0
2,Private room,0
3,Hotel room,0


In [42]:
SELECT
    COUNT(*) AS total_listings,
    COUNT(room_type) AS non_null_room_types,
    COUNT(*) - COUNT(room_type) AS null_room_types
FROM 'detailed_listings.csv';

,total_listings,non_null_room_types,null_room_types
0,4939,4939,0


In [45]:
/* Checking for hidden formatting inconsistences with room_type */ 

WITH trim_lower AS (
	SELECT 
		TRIM(room_type) AS trimmed_room_type,
		LOWER(room_type) AS lower_room_type
	FROM 'detailed_listings.csv'
)

SELECT 
	COUNT(DISTINCT trimmed_room_type) AS trim_count,
	COUNT(DISTINCT lower_room_type) AS lower_count
FROM trim_lower;

,trim_count,lower_count
0,4,4


In [46]:
SELECT
    COUNT(DISTINCT room_type) AS original_count,
    COUNT(DISTINCT LOWER(TRIM(room_type))) AS standardized_count
FROM 'detailed_listings.csv';

,original_count,standardized_count
0,4,4


## property_type

In [4]:
/* Finding the most common property types and listing the distinct property types */ 

SELECT 
	property_type,
	COUNT(*) AS listings
FROM 'detailed_listings.csv'
GROUP BY property_type
ORDER BY listings DESC;

,property_type,listings
0,Entire home,1589
1,Entire rental unit,1012
2,Entire condo,486
3,Private room in home,362
4,Entire guest suite,352
5,Entire townhouse,344
6,Entire guesthouse,294
7,Private room in townhouse,68
8,Room in hotel,65
9,Entire loft,60


In [32]:
/* Showing the rare categories of property types */ 

SELECT 
	property_type,
	COUNT(*) AS listings
FROM 'detailed_listings.csv'
GROUP BY property_type
HAVING COUNT(*) <= 5
ORDER BY listings DESC;

,property_type,listings
0,Private room in castle,5
1,Castle,4
2,Private room in guesthouse,4
3,Entire villa,3
4,Entire serviced apartment,2
5,Private room in hostel,2
6,Private room in casa particular,1
7,Treehouse,1
8,Entire vacation home,1
9,Casa particular,1


In [33]:
/* All the property types that contain 'Private room...' */

SELECT 
	property_type,
	COUNT(*) AS listings
FROM 'detailed_listings.csv'
WHERE property_type LIKE 'Private room%'
GROUP BY property_type
ORDER BY listings DESC;

,property_type,listings
0,Private room in home,362
1,Private room in townhouse,68
2,Private room in rental unit,50
3,Private room in bed and breakfast,26
4,Private room in guest suite,18
5,Private room in condo,15
6,Private room in bungalow,9
7,Private room in castle,5
8,Private room in guesthouse,4
9,Private room in hostel,2


In [2]:
/* All the property types that contain 'Room in...' */

SELECT 
	property_type,
	COUNT(*) AS listings
FROM 'detailed_listings.csv'
WHERE property_type LIKE 'Room in%' 
GROUP BY property_type
ORDER BY listings DESC;

,property_type,listings
0,Room in hotel,65
1,Room in aparthotel,19
2,Room in boutique hotel,10
3,Room in hostel,6


In [35]:
/* All the property types that contain 'Entire...' */

SELECT 
	property_type,
	COUNT(*) AS listings
FROM 'detailed_listings.csv'
WHERE property_type LIKE 'Entire%'
GROUP BY property_type
ORDER BY listings DESC;

,property_type,listings
0,Entire home,1589
1,Entire rental unit,1012
2,Entire condo,486
3,Entire guest suite,352
4,Entire townhouse,344
5,Entire guesthouse,294
6,Entire loft,60
7,Entire bungalow,44
8,Entire cottage,9
9,Entire place,9


In [38]:
/* Showing the rest of the property types that don't contain Private room, Room in, or Entire */

SELECT 
	property_type,
	COUNT(*) AS listings
FROM 'detailed_listings.csv'
WHERE 
	property_type NOT LIKE 'Private room%' AND property_type NOT LIKE 'Room in%' AND property_type NOT LIKE 'Entire%' 
GROUP BY property_type
ORDER BY listings DESC;

,property_type,listings
0,Shared room in home,36
1,Shared room in hostel,16
2,Tiny home,12
3,Castle,4
4,Farm stay,1
5,Treehouse,1
6,Casa particular,1


In [8]:
/* Investigating whether property_type actually provides information beyond room_type.
Within each broad room type, what kinds of actual properties make up that category? */ 

WITH room_property_type AS (
	SELECT
	    room_type,
	    property_type,
	    COUNT(*) AS listings,
	FROM 'detailed_listings.csv'
	GROUP BY room_type, property_type
), 

room_property_type_with_totals AS (
	SELECT room_type, property_type, listings,
		SUM(listings) OVER (
			PARTITION BY room_type
		) AS room_type_total
	FROM room_property_type
)

SELECT room_type, property_type, listings, room_type_total,
	ROUND(listings/room_type_total * 100, 2) AS percent_of_room_type_total
FROM room_property_type_with_totals
ORDER BY room_type, listings DESC;

/* 
- Entire home/apt - Entire home has the most listings at 1589 listings and 37.54% share of room type total. 
- I'm noticing there are private room types that could be better classified in 'Hotel room', including room in hotel, ...aparthotel, ...boutique hotel, ...hostel.
*/

,room_type,property_type,listings,room_type_total,percent_of_room_type_total
0,Entire home/apt,Entire home,1589,4233.0,37.54
1,Entire home/apt,Entire rental unit,1012,4233.0,23.91
2,Entire home/apt,Entire condo,486,4233.0,11.48
3,Entire home/apt,Entire guest suite,352,4233.0,8.32
4,Entire home/apt,Entire townhouse,344,4233.0,8.13
5,Entire home/apt,Entire guesthouse,294,4233.0,6.95
6,Entire home/apt,Entire loft,60,4233.0,1.42
7,Entire home/apt,Entire bungalow,44,4233.0,1.04
8,Entire home/apt,Tiny home,12,4233.0,0.28
9,Entire home/apt,Room in aparthotel,9,4233.0,0.21


## neighbourhood_cleansed

In [14]:
/* Checking for null rows */

SELECT 
	COUNT(*) AS total_rows,
	COUNT(neighbourhood_cleansed) AS non_null_rows,
	COUNT(*) - COUNT(neighbourhood_cleansed) AS null_rows
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows
0,4939,4939,0


In [15]:
/* Checking for formatting inconsistencies */ 

SELECT 
	COUNT(DISTINCT neighbourhood_cleansed) AS distinct_neighbourhood,
	COUNT(DISTINCT LOWER(TRIM(neighbourhood_cleansed))) AS lower_trip_neighbourhood
FROM 'detailed_listings.csv';

,distinct_neighbourhood,lower_trip_neighbourhood
0,77,77


In [39]:
/* Showing each neighbourhood and how many listings are in it */ 

WITH neighbourhood_listings AS (
	SELECT 
		neighbourhood_cleansed,
		COUNT(*) AS listings
	FROM 'detailed_listings.csv'
	GROUP BY neighbourhood_cleansed
	ORDER BY listings DESC
),

total_listings AS (
	SELECT 
		COUNT(*) AS total
	FROM 'detailed_listings.csv'
)

SELECT 
	n.neighbourhood_cleansed, n.listings,
	ROUND(n.listings/t.total * 100, 2) AS percent_of_total
FROM neighbourhood_listings AS n
CROSS JOIN total_listings AS t
ORDER BY percent_of_total DESC;

,neighbourhood_cleansed,listings,percent_of_total
0,Five Points,402,8.14
1,Highland,298,6.03
2,Union Station,226,4.58
3,Gateway - Green Valley Ranch,196,3.97
4,West Colfax,195,3.95
...,...,...,...
72,Marston,8,0.16
73,Wellshire,5,0.10
74,Auraria,5,0.10
75,Southmoor Park,4,0.08


In [40]:
/* Finding neighbourhoods represented by only a handful of listings */

SELECT 
	neighbourhood_cleansed,
	COUNT(*) AS listings
FROM 'detailed_listings.csv'
GROUP BY neighbourhood_cleansed
HAVING COUNT(*) <= 10
ORDER BY listings DESC;

,neighbourhood_cleansed,listings
0,Windsor,10
1,Harvey Park South,10
2,Fort Logan,9
3,Indian Creek,9
4,Marston,8
5,Auraria,5
6,Wellshire,5
7,Southmoor Park,4
8,Kennedy,1


In [13]:
/* Finding the most common room type and property type combination in each neighbourhood. Intentionally using RANK() to show ties. */ 

WITH neighbourhood_totals AS (
	SELECT 
		neighbourhood_cleansed,
		room_type,
		property_type,
		COUNT(*) AS listings
	FROM 'detailed_listings.csv'
	GROUP BY neighbourhood_cleansed, room_type, property_type
),

neighbourhood_totals_rank AS (
	SELECT 
		neighbourhood_cleansed,
		room_type,
		property_type,
		listings,
	 	RANK() OVER (
			PARTITION BY neighbourhood_cleansed
			ORDER BY listings DESC
		) AS listings_rank
	FROM neighbourhood_totals
)

SELECT * 
FROM neighbourhood_totals_rank
WHERE listings_rank = 1
ORDER BY listings DESC;

,neighbourhood_cleansed,room_type,property_type,listings,listings_rank
0,Five Points,Entire home/apt,Entire home,101,1
1,Union Station,Entire home/apt,Entire rental unit,95,1
2,CBD,Entire home/apt,Entire condo,90,1
3,Highland,Entire home/apt,Entire home,82,1
4,Gateway - Green Valley Ranch,Entire home/apt,Entire home,80,1
...,...,...,...,...,...
76,Bear Valley,Entire home/apt,Entire guest suite,3,1
77,Fort Logan,Entire home/apt,Entire home,3,1
78,Southmoor Park,Private room,Private room in home,2,1
79,Wellshire,Entire home/apt,Entire home,2,1


In [16]:
/* Do all 77 neighbourhood_cleansed values correspond to neighborhoods in neighbourhoods.csv, and does neighbourhoods.csv contain any neighborhoods with zero listings? 

listings neighbourhood -> Which don't exist in neighbourhoods.csv?

*/

SELECT 
	l.neighbourhood_cleansed,
	n.neighbourhood
FROM 'detailed_listings.csv' AS l
LEFT JOIN 'neighbourhoods.csv' AS n
	ON l.neighbourhood_cleansed = n.neighbourhood
WHERE n.neighbourhood IS NULL;

,neighbourhood_cleansed,neighbourhood


In [17]:
/* Do all 77 neighbourhood_cleansed values correspond to neighborhoods in neighbourhoods.csv, and does neighbourhoods.csv contain any neighborhoods with zero listings? 

neighbourhoods.csv -> Which don't appear in listings?

- Sun Valley is in neighbourhoods.csv but doesn't appear in detailed_listings.csv. This checks out as neighbourhoods.csv has +1 neighbourhood than listings at 78 instead of 77. 

*/

SELECT 
	n.neighbourhood,
	l.neighbourhood_cleansed
FROM 'neighbourhoods.csv' AS n
LEFT JOIN 'detailed_listings.csv' AS l
	ON n.neighbourhood = l.neighbourhood_cleansed
WHERE l.neighbourhood_cleansed IS NULL;

,neighbourhood,neighbourhood_cleansed
0,Sun Valley,None


## accommodates

In [24]:
/* Checking for null/zero values and Min, Max, Avg, Median */

SELECT 
	COUNT(*) AS total_rows,
	COUNT(accommodates) AS non_null_rows,
	COUNT(*) - COUNT(accommodates) AS null_rows,
	COUNT(CASE WHEN accommodates = 0 THEN 1 END) AS zero_values,
	MIN(accommodates) AS min_accommodates,
	MAX(accommodates) AS max_accommodates,
	AVG(accommodates) AS avg_accommodates,
	MEDIAN(accommodates) AS median_accommodates
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,min_accommodates,max_accommodates,avg_accommodates,median_accommodates
0,4939,4939,0,0,1,16,4.132618,4.0


In [29]:
SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	accommodates,
	bathrooms,
	bathrooms_text, 
	bedrooms
FROM 'detailed_listings.csv'
ORDER BY accommodates DESC;

,name,description,price,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms
0,Luxury house 6bedroom 4br and HOT TUB near to DIA,The whole group will be comfortable in this sp...,$537.00,Gateway - Green Valley Ranch,Entire home,Entire home/apt,16,3.5,3.5 baths,6
1,Beautiful Victorian home in charming cap hill,✨Charming Huge Home in Heart of Denver✨<br /><...,$683.33,Capitol Hill,Entire home,Entire home/apt,16,3.5,3.5 baths,7
2,"RiNo Reunion Retreat: Sleeps 50, 2 hot tubs, g...",🎉 Sleeps 50 Guests | 🛌 30 Beds | 💧 2 Hot Tubs ...,"$1,163.00",Five Points,Entire home,Entire home/apt,16,3.5,3.5 baths,9
3,3200 sqft • 5BRD • 9 beds • sleeps 16• Games,"This 3,200 sq foot home has 5 bedrooms 3.5 bat...",$505.47,Gateway - Green Valley Ranch,Entire home,Entire home/apt,16,3.5,3.5 baths,5
4,Denver Oasis · 5BR · 7-Seat Hot Tub · Sleeps 16,"Spacious 5BR/3BA Denver group home (~2,250 sq ...",$634.00,Elyria Swansea,Entire home,Entire home/apt,16,3.0,3 baths,5
...,...,...,...,...,...,...,...,...,...,...
4934,Alexandra's Queen Bed Room Long-term Rental Only,Thanks for visiting my Queen Bed Room for LONG...,$44.27,North Capitol Hill,Private room in home,Private room,1,1.0,1 private bath,<NA>
4935,Quiet and Relaxing City Living,"Come stay in this clean, comfortable and quiet...",$92.00,East Colfax,Private room in home,Private room,1,1.0,1 private bath,1
4936,Town Room 3,This room is very close to Downtown Denver,"$1,290.48",College View - South Platte,Private room in home,Private room,1,1.0,1 shared bath,<NA>
4937,Sloan's Lake Studio Cottage,This newly renovated cottage apartment has eve...,$61.88,Sloan Lake,Entire rental unit,Entire home/apt,1,1.0,1 bath,1


## bathrooms/bathrooms_text

In [25]:
/* Checking for null/zero values and Min, Max, Avg, Median */

SELECT 
	COUNT(*) AS total_rows,
	COUNT(bathrooms) AS non_null_rows,
	COUNT(*) - COUNT(bathrooms) AS null_rows,
	COUNT(CASE WHEN bathrooms = 0 THEN 1 END) AS zero_values,
	MIN(bathrooms) AS min_bathrooms,
	MAX(bathrooms) AS max_bathrooms,
	AVG(bathrooms) AS avg_bathrooms,
	MEDIAN(bathrooms) AS median_bathrooms
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,min_bathrooms,max_bathrooms,avg_bathrooms,median_bathrooms
0,4939,4052,887,0,0.5,17.0,1.530972,1.0


In [30]:
SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	accommodates,
	bathrooms,
	bathrooms_text, 
	bedrooms
FROM 'detailed_listings.csv'
WHERE bathrooms IS NULL;

/* 
- bathrooms_text gives context to many records that have null bathrooms, so it cannot be assumed that all null bathrooms listings have zero bathrooms. 
- It's worth noting that many null bathrooms listings also have null prices
*/

,name,description,price,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms
0,Lodo / RiNo LOFT via airport train,"Modern 1,000 square foot loft in the heart of ...",$32.35,Five Points,Entire loft,Entire home/apt,3,NaN,1.5 baths,1
1,a REAL warehouse LOFT,two level LOFT.. great vibe.<br />spiral stair...,$25.66,Five Points,Entire loft,Entire home/apt,4,NaN,1 bath,2
2,Quiet Highlands Cottage,Located in the Historic Landmark District of W...,None,West Highland,Entire guest suite,Entire home/apt,2,NaN,1 bath,1
3,"2 ZEN! bright,warm,cozy, 5% off 6 Months,NOsmo...",Our cozy and comfortable one bedroom apartment...,$72.68,City Park West,Entire rental unit,Entire home/apt,1,NaN,1 bath,1
4,Cozy Bungalow in Quiet and Serene Neighborhood,This is for the entire top floor unit of our 2...,$82.78,Sunnyside,Entire home,Entire home/apt,4,NaN,1 bath,2
...,...,...,...,...,...,...,...,...,...,...
882,The Denver Dorms,The Denver Dorms <br />Designed by someone who...,$15.20,Overland,Shared room in home,Shared room,1,NaN,1 shared bath,1
883,Denver Airport 15 minutes away. One king bed.,Relax with the whole family at this peaceful p...,$95.86,Gateway - Green Valley Ranch,Private room in guest suite,Private room,2,NaN,1 shared bath,1
884,Travel Nurse & Corporate Housing Near DIA 3BR,"Looking for a comfortable, fully furnished hom...",$122.09,Gateway - Green Valley Ranch,Entire home,Entire home/apt,5,NaN,2.5 baths,3
885,Pathy Airbnb,Take a break and unwind at this peaceful and ...,$245.50,Gateway - Green Valley Ranch,Entire home,Entire home/apt,7,NaN,2.5 baths,3


In [31]:
/* Listings where both bathrooms and bathrooms_text are NULL */ 

SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	accommodates,
	bathrooms,
	bathrooms_text, 
	bedrooms
FROM 'detailed_listings.csv'
WHERE bathrooms IS NULL AND bathrooms_text IS NULL;

,name,description,price,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms
0,Ballarat Chill Apartment In DTC With A Pool & ...,Hello !! I’m Debbie and I want to host you at ...,$105.31,Southmoor Park,Private room in serviced apartment,Private room,2,NaN,None,1
1,Modern Flat w/ Mountain Views + Private Balcony,Welcome to The Berkeley Hotel<br /><br />Welco...,$550.00,Berkeley,Room in hotel,Hotel room,6,NaN,None,2


In [37]:
SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	accommodates,
	bathrooms,
	bathrooms_text, 
	bedrooms
FROM 'detailed_listings.csv'
WHERE bathrooms IS NOT NULL AND bathrooms_text IS NULL;

,name,description,price,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms
0,Shared Room - 1 twin bed in 12 bunk bed co-ed ...,Get ready to kick back and enjoy the fanciest ...,$44.97,Capitol Hill,Room in hostel,Hotel room,1,17.0,None,1
1,Stylish Getaway w/ Balcony & Scenic Mountain V...,Welcome to The Berkeley Hotel!<br /><br />Step...,$591.00,Berkeley,Room in hotel,Hotel room,6,2.0,None,2
2,Sleek Tennyson St Flat w/ Balcony & Views,Welcome to The Berkeley Hotel! <br /><br />Thi...,$512.00,Berkeley,Room in hotel,Hotel room,6,2.0,None,2
3,Luxurious Flat w/ Private Balcony & Mtn Views!,Welcome to The Berkeley Hotel! <br /><br />Thi...,$546.00,Berkeley,Room in hotel,Hotel room,6,2.0,None,2


In [34]:
SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	accommodates,
	bathrooms,
	bathrooms_text, 
	bedrooms
FROM 'detailed_listings.csv'
ORDER BY bathrooms DESC
LIMIT 10;

,name,description,price,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms
0,Shared room - 1 Twin Bed in 4 bed Co-Ed Dorm,Get ready to kick back and enjoy the fanciest ...,$60.14,Capitol Hill,Shared room in hostel,Shared room,1,17.0,17 shared baths,1
1,Shared Room - 1 twin bed in 12 bunk bed co-ed ...,Get ready to kick back and enjoy the fanciest ...,$44.97,Capitol Hill,Room in hostel,Hotel room,1,17.0,None,1
2,Shared Room - 1 twin bed in 16 bunk bed co-ed ...,Get ready to kick back and enjoy the fanciest ...,$42.84,Capitol Hill,Shared room in hostel,Shared room,16,11.0,11 shared baths,1
3,The Curtis Park Club | Ultimate Group Retreat,The Curtis Park Club is an intimate boutique h...,"$1,814.00",Five Points,Entire rental unit,Entire home/apt,16,10.0,10 baths,10
4,Private Ensuite in a Historic Hostel in Denver,Stay in our lovingly restored Hotel-turned-Hos...,$179.00,Capitol Hill,Room in hotel,Private room,2,9.0,9 baths,1
5,The Castle Marne,The Castle Marne is one of Denver's most iconi...,"$2,995.00",City Park West,Castle,Entire home/apt,10,8.0,8 baths,5
6,Snazzy Denver Stay for Friends and Family!,"A snazzy private room with shared bathroom, wi...",$135.00,Capitol Hill,Room in hotel,Private room,6,8.0,8 baths,1
7,Modern Sanctuary: historic church steps from RiNo,Gather your group in one of Denver’s most spec...,"$1,641.00",Whittier,Castle,Entire home/apt,16,6.5,6.5 baths,6
8,"Huge LUX Family Home with Roof Deck, Chef's Dream",Effortless elegance is envisioned in this stun...,"$1,336.80",Hilltop,Entire home,Entire home/apt,6,6.5,6.5 baths,4
9,Maison Monroe,"An elegant blend of comfort and grandeur, this...","$1,152.50",University Park,Entire villa,Entire home/apt,14,6.0,6 baths,4


In [32]:
/* What values occur in bathrooms_text, especially when bathrooms is NULL? */ 

SELECT 
	bathrooms_text,
	COUNT(*) AS listings
FROM 'detailed_listings.csv'
WHERE bathrooms IS NULL
GROUP BY bathrooms_text
ORDER BY listings DESC;

,bathrooms_text,listings
0,1 bath,430
1,2 baths,148
2,2.5 baths,62
3,1.5 baths,51
4,1 private bath,49
5,3 baths,36
6,3.5 baths,30
7,1 shared bath,23
8,2 shared baths,13
9,4 baths,9


In [35]:
/* When bathrooms is not NULL, does bathrooms_text generally describe the same numeric count? */

SELECT 
	bathrooms,
	bathrooms_text,
	COUNT(*) AS listings
FROM 'detailed_listings.csv'
WHERE bathrooms IS NOT NULL AND bathrooms_text IS NOT NULL 
GROUP BY bathrooms, bathrooms_text
ORDER BY listings DESC;

/*
- For the most part, the numeric meanings in bathrooms and bathrooms_text line up when both are NOT NULL. The one exception I'm seeing is one record that lists 2 bathrooms but 3.5 baths in bathrooms_text. 
*/

,bathrooms,bathrooms_text,listings
0,1.0,1 bath,2134
1,2.0,2 baths,571
2,1.0,1 private bath,293
3,2.5,2.5 baths,282
4,1.5,1.5 baths,176
5,1.0,1 shared bath,152
6,3.5,3.5 baths,141
7,3.0,3 baths,137
8,4.0,4 baths,47
9,1.5,1.5 shared baths,35


In [36]:
SELECT 
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	accommodates,
	bathrooms,
	bathrooms_text, 
	bedrooms
FROM 'detailed_listings.csv'
WHERE bathrooms = 2 AND bathrooms_text = '3.5 baths';

,name,description,price,neighbourhood_cleansed,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms
0,"LoHi Luxury – Eat, Drink & Explore Denver’s Best","Stay in the heart of LoHi, Denver’s most walka...",$382.00,Highland,Entire townhouse,Entire home/apt,4,2.0,3.5 baths,2


## bedrooms/beds

In [38]:
/* Checking for null/zero values and Min, Max, Avg, Median */

SELECT 
	COUNT(*) AS total_rows,
	COUNT(bedrooms) AS non_null_rows,
	COUNT(*) - COUNT(bedrooms) AS null_rows,
	COUNT(CASE WHEN bedrooms = 0 THEN 1 END) AS zero_values,
	MIN(bedrooms) AS min_bedrooms,
	MAX(bedrooms) AS max_bedrooms,
	AVG(bedrooms) AS avg_bedrooms,
	MEDIAN(bedrooms) AS median_bedrooms
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,min_bedrooms,max_bedrooms,avg_bedrooms,median_bedrooms
0,4939,4392,547,0,1,15,1.947177,2.0


In [46]:
SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	accommodates,
	bedrooms,
	beds,
	bathrooms,
	bathrooms_text
FROM 'detailed_listings.csv'
WHERE bedrooms IS NULL;

,name,description,price,neighbourhood_cleansed,property_type,room_type,accommodates,bedrooms,beds,bathrooms,bathrooms_text
0,private,This room is in the basement. It does not hav...,$80.00,North Park Hill,Private room in home,Private room,2,<NA>,1,1.0,1 shared bath
1,Alexandra's Queen Bed Room Long-term Rental Only,Thanks for visiting my Queen Bed Room for LONG...,$44.27,North Capitol Hill,Private room in home,Private room,1,<NA>,2,1.0,1 private bath
2,"Baker Studio Pet-friendly, Walkable, Patio",Private studio retreat in a charming historic ...,$191.72,Baker,Entire guesthouse,Entire home/apt,2,<NA>,1,1.0,1 bath
3,Treetop View Room East City Park 1-2 months,Located in the desirable district neighborhood...,$35.47,South Park Hill,Private room in rental unit,Private room,1,<NA>,1,1.0,1 shared bath
4,Private room in the nest - w/bathroom,Private small room with own bath on the 2nd fl...,$102.71,Baker,Private room in townhouse,Private room,4,<NA>,1,1.0,1 private bath
...,...,...,...,...,...,...,...,...,...,...,...
542,"Luxury Cherry Creek Studio | Pool, Gym, Parking",Luxury furnished studio in Cherry Creek Denver...,$125.76,Cherry Creek,Entire rental unit,Entire home/apt,2,<NA>,1,1.0,1 bath
543,"Cherry Creek North, Balcony, Rooftop Pool, Par...","Wake up above Cherry Creek in a light-filled, ...",$134.52,Cherry Creek,Entire rental unit,Entire home/apt,2,<NA>,1,1.0,1 bath
544,Healthcare Heroes Welcome! Mid-Term Stay in De...,Dog Friendly w/ Fee | Easy Hospital Access | 1...,$70.38,City Park West,Entire loft,Entire home/apt,2,<NA>,2,1.0,1 bath
545,Urban Oasis in Cap Hill,"Bright, cozy retreat in a historic Denver bric...",$58.80,Capitol Hill,Entire rental unit,Entire home/apt,2,<NA>,1,1.0,1 bath


In [43]:
SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	accommodates,
	bedrooms,
	beds,
	bathrooms,
	bathrooms_text
FROM 'detailed_listings.csv'
ORDER BY bedrooms DESC
LIMIT 20;

/* 
- 15 bedrooms can be seen as an extreme value since the next highest amount of bedrooms is 10. The 15-bedroom listing is worth investigating because it accommodates much less people than the listings below it and is at a significantly lower price. However, it is still a legitimate listings based on the corresponding fields. It's also worth noting that it lists 50 beds, while only accommodating 2. 

- I would flag that record as a suspected data-quality anomaly, rather than automatically calling it incorrect. You don't yet have enough evidence to overwrite or remove it. That's a useful distinction for a portfolio project.
*/

,name,description,price,neighbourhood_cleansed,property_type,room_type,accommodates,bedrooms,beds,bathrooms,bathrooms_text
0,Luxury Location Lodo,Elevate your Denver experience from this high-...,$164.11,Union Station,Entire rental unit,Entire home/apt,2,15,50,2.5,2.5 baths
1,The Curtis Park Club | Ultimate Group Retreat,The Curtis Park Club is an intimate boutique h...,"$1,814.00",Five Points,Entire rental unit,Entire home/apt,16,10,10,10.0,10 baths
2,Biggest Mansion in denver downtown. 5369 feet,"Gigantic Home, Very Clean and comfortable with...",$669.87,West Colfax,Entire home,Entire home/apt,16,9,17,5.0,5 baths
3,"RiNo Reunion Retreat: Sleeps 50, 2 hot tubs, g...",🎉 Sleeps 50 Guests | 🛌 30 Beds | 💧 2 Hot Tubs ...,"$1,163.00",Five Points,Entire home,Entire home/apt,16,9,30,3.5,3.5 baths
4,"Large Groups ok! Hot Tub, Pickleball, Free Par...",Bring the whole family to this great place wit...,"$1,116.52",University Hills,Entire home,Entire home/apt,16,8,13,6.0,6 baths
5,Unique Historic Victorian Mansion near Downtown,"Welcome to the Chamberlain Mansion, an extraor...",$726.50,City Park West,Entire home,Entire home/apt,16,8,11,4.0,4 baths
6,Downtown's Largest Rental - Walk to Everything!,Welcome to Hotel Residential of Denver.<br /><...,"$1,988.62",Five Points,Entire condo,Entire home/apt,16,8,18,4.0,4 baths
7,7Bedrooms/14Beds Gated Parking Contractor Special,💨 KushKations Gameroom: 420 friendly 7BR/2BA w...,$239.51,Clayton,Entire home,Entire home/apt,16,7,14,2.0,2 baths
8,Mini Mansion: A+ Location - Denver Highlands,"5 minute walk to Sushi, sports bar, amazing w...",$523.33,West Highland,Entire home,Entire home/apt,15,7,9,3.5,3.5 baths
9,"Deluxe Denver Retreat: Lake, Golf & Mountain V...",Escape to Serenity: A Luxurious Denver Retreat...,$775.43,Gateway - Green Valley Ranch,Entire home,Entire home/apt,16,7,9,4.0,4 baths


In [44]:
/* Checking for null/zero values and Min, Max, Avg, Median */

SELECT 
	COUNT(*) AS total_rows,
	COUNT(beds) AS non_null_rows,
	COUNT(*) - COUNT(beds) AS null_rows,
	COUNT(CASE WHEN beds = 0 THEN 1 END) AS zero_values,
	MIN(beds) AS min_beds,
	MAX(beds) AS max_beds,
	AVG(beds) AS avg_beds,
	MEDIAN(beds) AS median_beds
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,min_beds,max_beds,avg_beds,median_beds
0,4939,4211,728,0,1,50,2.310615,2.0


In [45]:
SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	accommodates,
	bedrooms,
	beds,
	bathrooms,
	bathrooms_text
FROM 'detailed_listings.csv'
WHERE bedrooms IS NULL AND beds IS NULL
LIMIT 20;

,name,description,price,neighbourhood_cleansed,property_type,room_type,accommodates,bedrooms,beds,bathrooms,bathrooms_text
0,Private Denver condo near Capitol,Great little home base in the middle of Capito...,$66.40,Capitol Hill,Entire condo,Entire home/apt,2,<NA>,<NA>,1.0,1 bath
1,Large Bedrm Gym Views PvtBath Prkg,Enjoy the view from a large comfy bedroom in a...,$9.66,Capitol Hill,Private room in rental unit,Private room,2,<NA>,<NA>,NaN,1 private bath
2,Amazing Bunk Female Only Shared Room!,Our FEMALE ONLY shared room accommodations are...,None,Five Points,Shared room in hostel,Shared room,6,<NA>,<NA>,NaN,2 shared baths
3,Amazing View of MTS in 4-Person Mixed Room!,Take in stunning views of the mountains from t...,None,Five Points,Shared room in hostel,Shared room,4,<NA>,<NA>,NaN,2 shared baths
4,Downtown Apt - Walker's Paradise! Brooks Tower 4I,"Located in downtown Denver, unit 4I is in the ...",$72.79,CBD,Entire condo,Entire home/apt,2,<NA>,<NA>,1.0,1 bath
5,"Denver, Berkeley, Regis Area, Private Studio S...",Street Level Artist Studio Apt. (600 SQ FT) Fr...,$95.50,Regis,Entire guest suite,Entire home/apt,2,<NA>,<NA>,NaN,1 bath
6,Quaint Bedroom in heart of Denver w/ private bath,Comfortable basement room with an egress windo...,None,Rosedale,Private room in home,Private room,2,<NA>,<NA>,NaN,1 private bath
7,RiNo Oasis in Historic Area - Walk Everywhere!,Private room located downtown 8 blocks from Co...,None,Five Points,Private room in condo,Private room,2,<NA>,<NA>,NaN,1 shared bath
8,"Large, Sunny Space in the Heart of Denver",A large private bedroom with private bathroom....,None,Rosedale,Private room in home,Private room,2,<NA>,<NA>,NaN,1 private bath
9,Cozy room near Sloan’s Lake with rooftop!,Hello everyone! Offered is a bedroom in my 191...,None,West Colfax,Private room in home,Private room,1,<NA>,<NA>,NaN,1 shared bath


In [2]:
/* 

bedrooms populated | beds populated
bedrooms populated | beds NULL
bedrooms NULL      | beds populated
bedrooms NULL      | beds NULL

*/

SELECT 
	COUNT(CASE WHEN bedrooms IS NOT NULL AND beds IS NOT NULL THEN 1 END) AS bedrooms_populated_bed_populated,
	COUNT(CASE WHEN bedrooms IS NOT NULL AND beds IS NULL THEN 1 END) AS bedrooms_populated_beds_null,
	COUNT(CASE WHEN bedrooms IS NULL AND beds IS NOT NULL THEN 1 END) AS bedrooms_null_beds_populated,
	COUNT(CASE WHEN bedrooms IS NULL AND beds IS NULL THEN 1 END) AS bedrooms_null_beds_null
FROM 'detailed_listings.csv';

,bedrooms_populated_bed_populated,bedrooms_populated_beds_null,bedrooms_null_beds_populated,bedrooms_null_beds_null
0,3735,657,476,71


In [51]:
/* For listings where both fields exist, how does beds generally relate to bedrooms? */ 

SELECT 
	bedrooms || ' ' || 'bedroom' AS bedrooms,
	beds || ' ' || 'bed' AS beds,
	COUNT(*) AS listings
FROM 'detailed_listings.csv'
WHERE bedrooms IS NOT NULL AND beds IS NOT NULL
GROUP BY bedrooms, beds
ORDER BY listings DESC;

,bedrooms,beds,listings
0,1 bedroom,1 bed,1201
1,2 bedroom,2 bed,840
2,1 bedroom,2 bed,380
3,3 bedroom,3 bed,315
4,2 bedroom,3 bed,232
...,...,...,...
62,8 bedroom,11 bed,1
63,2 bedroom,12 bed,1
64,5 bedroom,12 bed,1
65,7 bedroom,14 bed,1


In [3]:
/* beds/bedrooms */

SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	ROUND(beds/bedrooms, 2) AS bed_to_bedroom_ratio,
	bedrooms,
	beds,
	accommodates
FROM 'detailed_listings.csv'
WHERE bedrooms IS NOT NULL AND beds IS NOT NULL
ORDER BY bed_to_bedroom_ratio DESC;

,name,description,price,neighbourhood_cleansed,property_type,room_type,bed_to_bedroom_ratio,bedrooms,beds,accommodates
0,Rustic treehouse retreat in the city.,Creative compound with 3 efficiency spaces: ca...,$109.90,Regis,Treehouse,Entire home/apt,12.00,1,12,2
1,Bed in Fantastic Shared Room Downtown,Our shared rooms are all stocked full of uniqu...,$43.00,Five Points,Shared room in hostel,Shared room,10.00,1,10,1
2,Stellar Shared Room Next to Best Bars in Denver!,Our shared room accommodations are all stocked...,$47.00,Five Points,Shared room in hostel,Shared room,10.00,1,10,8
3,Bed in Shared Room in Heart of Downtown!,Our shared rooms are all stocked full of uniqu...,$47.00,Five Points,Shared room in hostel,Shared room,10.00,1,10,1
4,Stay Close to Union Station in Downtown Denver!,Our shared room accommodations are all stocked...,$44.00,Five Points,Shared room in hostel,Shared room,10.00,1,10,10
...,...,...,...,...,...,...,...,...,...,...
3730,SkyLoft: A Unique Space in Denver,Welcome to SkyLoft! Whether you're visiting De...,$175.00,Skyland,Entire guesthouse,Entire home/apt,0.50,2,1,2
3731,"Denver Gem, Split Home, Central",Enjoy your time in the Mile-high city staying ...,None,Whittier,Private room in home,Private room,0.33,3,1,2
3732,Cozy RiNo Bungalow - Easy Car/Bus ride to Boulder,Cozy house located just a few blocks from RiNo...,$328.25,Cole,Entire home,Entire home/apt,0.33,3,1,4
3733,Town Room1,The space is just 5 minutes to Downtown Denver...,None,College View - South Platte,Private room in home,Private room,0.33,3,1,2


In [7]:
/* beds/bedrooms */

SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	ROUND(beds/bedrooms, 2) AS bed_to_bedroom_ratio,
	ROUND(beds/accommodates, 2) AS beds_accommodates_ratio,
	ROUND(bedrooms/accommodates, 2) AS bedrooms_accommodates_ratio,
	bedrooms,
	beds,
	accommodates
FROM 'detailed_listings.csv'
WHERE beds > accommodates
ORDER BY beds_accommodates_ratio DESC;

,name,description,price,neighbourhood_cleansed,property_type,room_type,bed_to_bedroom_ratio,beds_accommodates_ratio,bedrooms_accommodates_ratio,bedrooms,beds,accommodates
0,Luxury Location Lodo,Elevate your Denver experience from this high-...,$164.11,Union Station,Entire rental unit,Entire home/apt,3.33,25.00,7.50,15,50,2
1,Bed in Fantastic Shared Room Downtown,Our shared rooms are all stocked full of uniqu...,$43.00,Five Points,Shared room in hostel,Shared room,10.00,10.00,1.00,1,10,1
2,Bed in Shared Room in Heart of Downtown!,Our shared rooms are all stocked full of uniqu...,$47.00,Five Points,Shared room in hostel,Shared room,10.00,10.00,1.00,1,10,1
3,The Druid Dorms 3,The Druid Dorms<br />Nestled by the gentle flo...,$17.76,Overland,Shared room in home,Shared room,6.00,6.00,1.00,1,6,1
4,Rustic treehouse retreat in the city.,Creative compound with 3 efficiency spaces: ca...,$109.90,Regis,Treehouse,Entire home/apt,12.00,6.00,0.50,1,12,2
5,Green valley ranch; Private Room; Denver Airport,- Quiet and friendly neighborhood<br />- 15 mi...,None,Gateway - Green Valley Ranch,Private room in home,Private room,NaN,4.00,NaN,<NA>,4,1
6,comfort place in Denver,very close to Denver International Airport dow...,$59.97,Gateway - Green Valley Ranch,Entire home,Entire home/apt,5.00,2.50,0.50,1,5,2
7,Stand-Alone Cottage w/ Fenced Patio & Yard Oasis,Enjoy comfort & style in our thoughtfully desi...,$136.81,Berkeley,Entire guesthouse,Entire home/apt,NaN,2.33,NaN,<NA>,7,3
8,BRAND NEW apt. w/full kitchen and full gym,Brand new remodeled space in the University Hi...,$78.17,University Hills,Entire guest suite,Entire home/apt,4.00,2.00,0.50,1,4,2
9,5. Monaco bedroom w/ j&k bath,Beautiful house. This is for one individual be...,None,Congress Park,Private room in home,Private room,1.00,2.00,2.00,2,2,1


In [13]:
/* Median beds per bedroom
Average beds per bedroom
Q1
Q3
Maximum

possibly broken down by room_type */

WITH bed_ratios AS (
	SELECT 
		room_type,
		beds/bedrooms AS beds_per_bedroom
	FROM 'detailed_listings.csv'
	WHERE bedrooms IS NOT NULL AND beds IS NOT NULL AND bedrooms > 0
)

SELECT 
	room_type,
	COUNT(*) AS listing_count,
	ROUND(AVG(beds_per_bedroom), 2) AS avg_beds_per_bedroom,
	QUANTILE_CONT(beds_per_bedroom, 0.25) AS q1_beds_per_bedroom,
	MEDIAN(beds_per_bedroom) AS median_beds_per_bedroom,
	QUANTILE_CONT(beds_per_bedroom, 0.75) AS q3_beds_per_bedroom,
	MAX(beds_per_bedroom) AS max_beds_per_bedroom
FROM bed_ratios
GROUP BY room_type
ORDER BY listing_count DESC;

,room_type,listing_count,avg_beds_per_bedroom,q1_beds_per_bedroom,median_beds_per_bedroom,q3_beds_per_bedroom,max_beds_per_bedroom
0,Entire home/apt,3431,1.26,1.000,1.0,1.400,12.0
1,Private room,235,1.17,1.000,1.0,1.000,8.0
2,Shared room,49,2.72,1.000,1.0,2.000,10.0
3,Hotel room,20,1.50,1.375,1.5,1.625,2.0


## minimum_nights

In [21]:
/* Checking for nulls, min, max, median, avg, distribution, and extremely high values */

SELECT 
	COUNT(*) AS total_rows,
	COUNT(minimum_nights) AS non_null_rows,
	COUNT(*) - COUNT(minimum_nights) AS null_values,
	COUNT(CASE WHEN minimum_nights = 0 THEN 1 END) AS zero_values, 
	MIN(minimum_nights) AS min_minimum_nights,
	MAX(minimum_nights) AS max_minimum_nights,
	ROUND(AVG(minimum_nights), 2) AS avg_minimum_nights,
	QUANTILE_CONT(minimum_nights, 0.25) AS q1_minimum_nights,
	MEDIAN(minimum_nights) AS median_minimum_nights,
	QUANTILE_CONT(minimum_nights, 0.75) AS q3_minimum_nights
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_values,zero_values,min_minimum_nights,max_minimum_nights,avg_minimum_nights,q1_minimum_nights,median_minimum_nights,q3_minimum_nights
0,4939,4938,1,0,1,500,17.4,1.0,7.0,29.0


In [5]:
/* Investigating listings with minimum nights over q3 (29 nights) */

SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	minimum_nights,
	maximum_nights
FROM 'detailed_listings.csv'
WHERE minimum_nights > 29
ORDER BY minimum_nights DESC, maximum_nights DESC;

,name,description,price,neighbourhood_cleansed,property_type,room_type,minimum_nights,maximum_nights
0,"Modern apartment, cool neighborhood","Modern apartment with wood and tile floors, on...",None,Congress Park,Entire rental unit,Entire home/apt,500,1125
1,Welcoming Family friendly home,"Our property is close to Airport, Aurora medic...",None,Gateway - Green Valley Ranch,Entire home,Entire home/apt,365,1125
2,"Colorful 2bd, Best Location!",Relax with the whole family at this peaceful p...,None,Platt Park,Entire home,Entire home/apt,365,700
3,Urban Oasis in LoHi Neighborhood,This fantastic apartment seamlessly combines s...,$7.38,Highland,Entire guesthouse,Entire home/apt,365,365
4,"Landing | Stylish Studio, Clubhouse",Settle into an upscale stay in Denver’s most w...,None,Cherry Creek,Entire rental unit,Entire home/apt,365,365
...,...,...,...,...,...,...,...,...
1116,Loft style LoDo Condo/30 Day Rental,Beautiful condominium with open floor plan. Th...,None,Union Station,Entire condo,Entire home/apt,30,35
1117,City View House 2BED/2BATH 30+ Days,Enjoy a stylish experience at this centrally-l...,$175.85,Union Station,Entire condo,Entire home/apt,30,30
1118,Sunny ranch house,None,None,Virginia Village,Entire home,Entire home/apt,30,30
1119,"Beautiful, centrally located- 3 bed/2 bath",Spacious home with an amazing yard and back de...,None,University,Entire home,Entire home/apt,30,30


In [2]:
/* Investigating null value for minimum nights 
- Both minimum and maximum nights are null for this listing. 
*/ 

SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	minimum_nights,
	maximum_nights
FROM 'detailed_listings.csv'
WHERE minimum_nights IS NULL;

,name,description,price,neighbourhood_cleansed,property_type,room_type,minimum_nights,maximum_nights
0,Private Suite | Own Entrance & Bath | Kitchenette,"Your own private entrance, private bath, and b...",$181.00,Clayton,Entire home,Entire home/apt,<NA>,<NA>


In [19]:
/* Looking into minimum night statistics by room type */ 

SELECT 
	room_type,
	COUNT(*) AS listings,
	ROUND(AVG(minimum_nights), 2) AS avg_minimum_nights_by_room_type,
	MEDIAN(minimum_nights) AS median_minimum_nights_by_room_type,
	MIN(minimum_nights) AS min_minimum_nights_by_room_type,
	MAX(minimum_nights) AS max_minimum_nights_by_room_type
FROM 'detailed_listings.csv'
GROUP BY room_type;

,room_type,listings,avg_minimum_nights_by_room_type,median_minimum_nights_by_room_type,min_minimum_nights_by_room_type,max_minimum_nights_by_room_type
0,Shared room,52,17.06,29.0,1,30
1,Entire home/apt,4233,17.79,29.0,1,500
2,Private room,631,15.22,3.0,1,180
3,Hotel room,23,5.87,1.0,1,29


In [18]:
/* Looking at the distribution of listing counts by minimum nights 
- 1 minimum night has the dominates the share of listings, but 29 and 30 minimum nights also have substantial shares of the listings. 
*/ 

WITH minimum_nights_distribution AS (
	SELECT 
		minimum_nights || ' ' || 'nights' AS minimum_nights,
		COUNT(*) AS listings
	FROM 'detailed_listings.csv'
	GROUP BY minimum_nights
),

total_nights AS (
	SELECT 
		COUNT(*) AS total
	FROM 'detailed_listings.csv'
)

SELECT 
	m.minimum_nights, m.listings,
	ROUND(m.listings/t.total * 100, 2) AS percent_of_total
FROM minimum_nights_distribution AS m
CROSS JOIN total_nights AS t
ORDER BY percent_of_total DESC;

,minimum_nights,listings,percent_of_total
0,1 nights,1337,27.07
1,29 nights,1324,26.81
2,30 nights,991,20.06
3,2 nights,857,17.35
4,3 nights,204,4.13
5,4 nights,38,0.77
6,90 nights,32,0.65
7,60 nights,23,0.47
8,31 nights,23,0.47
9,7 nights,20,0.40


In [20]:
/* Looking at minimum nights by neighbourhood */ 

SELECT 
	neighbourhood_cleansed,
	COUNT(*) AS listings,
	ROUND(AVG(minimum_nights), 2) AS avg_minimum_nights_by_neighbourhood,
	MEDIAN(minimum_nights) AS median_minimum_nights_by_neighbourhood,
	MIN(minimum_nights) AS min_minimum_nights_by_neighbourhood,
	MAX(minimum_nights) AS max_minimum_nights_by_neighbourhood
FROM 'detailed_listings.csv'
GROUP BY neighbourhood_cleansed
ORDER BY avg_minimum_nights_by_neighbourhood DESC;

,neighbourhood_cleansed,listings,avg_minimum_nights_by_neighbourhood,median_minimum_nights_by_neighbourhood,min_minimum_nights_by_neighbourhood,max_minimum_nights_by_neighbourhood
0,Southmoor Park,4,53.00,59.5,3,90
1,Windsor,10,33.40,29.0,1,180
2,Kennedy,1,31.00,31.0,31,31
3,Cherry Creek,67,30.63,3.0,1,365
4,Indian Creek,9,29.67,30.0,29,30
...,...,...,...,...,...,...
72,Whittier,108,9.63,2.0,1,45
73,North Park Hill,61,9.57,2.0,1,30
74,Wellshire,5,6.80,1.0,1,29
75,Civic Center,42,6.62,1.0,1,30


In [24]:
/* Creating minimum-stay segments */ 

WITH minimum_stay_segmentation AS (
	SELECT 
		minimum_nights,
		CASE
			WHEN minimum_nights = 1 THEN '1 night'
			WHEN minimum_nights <= 6 THEN '2-6 nights'
			WHEN minimum_nights <= 28 THEN '7-28 nights'
			WHEN minimum_nights <= 30 THEN '29-30 nights'
			WHEN minimum_nights <= 89 THEN '31-89 nights'
			WHEN minimum_nights IS NULL THEN 'Missing'
			ELSE '90+ nights'
		END AS minimum_stay_segments
	FROM 'detailed_listings.csv'
),

total_listings AS (
	SELECT 
		COUNT(*) AS total
	FROM 'detailed_listings.csv'
),

count_minimum_stay_segments AS (
	SELECT 
		minimum_stay_segments,
		COUNT(*) AS listings
	FROM minimum_stay_segmentation
	GROUP BY minimum_stay_segments
)

SELECT 
	c.minimum_stay_segments, c.listings,
	ROUND(c.listings/t.total * 100, 2) AS percent_of_total
FROM count_minimum_stay_segments AS c
CROSS JOIN total_listings AS t
ORDER BY percent_of_total DESC;

,minimum_stay_segments,listings,percent_of_total
0,29-30 nights,2315,46.87
1,1 night,1337,27.07
2,2-6 nights,1120,22.68
3,90+ nights,66,1.34
4,31-89 nights,64,1.30
5,7-28 nights,36,0.73
6,Missing,1,0.02


## availability

In [27]:
/* Checks for availability_30 */ 

SELECT
	COUNT(*) AS total_rows,
	COUNT(availability_30) AS non_null_rows,
	COUNT(*) - COUNT(availability_30) AS null_rows,
	COUNT(CASE WHEN availability_30 = 0 THEN 1 END) AS zero_values,
	COUNT(CASE WHEN availability_30 > 30 THEN 1 END) AS over_30_values,
	ROUND(AVG(availability_30), 2) AS avg_availability_30,
	MEDIAN(availability_30) AS median_availability_30,
	MIN(availability_30) AS min_availability_30, 
	MAX(availability_30) AS max_availability_30
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,over_30_values,avg_availability_30,median_availability_30,min_availability_30,max_availability_30
0,4939,4939,0,1734,0,9.04,6.0,0,30


In [38]:
/* Distribution for availability_30 */ 

WITH listings AS (
	SELECT 
		availability_30,
		COUNT(*) AS listings
	FROM 'detailed_listings.csv'
	GROUP BY availability_30
	ORDER BY listings DESC
),

total_listings AS (
	SELECT 
		COUNT(*) AS total
	FROM 'detailed_listings.csv'
)

SELECT 
	l.availability_30, l.listings,
	ROUND(l.listings/t.total * 100, 2) AS percent_of_total
FROM listings AS l
CROSS JOIN total_listings AS t
ORDER BY percent_of_total DESC;

,availability_30,listings,percent_of_total
0,0,1734,35.11
1,30,222,4.49
2,1,152,3.08
3,9,152,3.08
4,2,151,3.06
5,3,149,3.02
6,5,148,3.00
7,7,139,2.81
8,6,135,2.73
9,8,133,2.69


In [28]:
/* Checks for availability_60 */ 

SELECT
	COUNT(*) AS total_rows,
	COUNT(availability_60) AS non_null_rows,
	COUNT(*) - COUNT(availability_60) AS null_rows,
	COUNT(CASE WHEN availability_60 = 0 THEN 1 END) AS zero_values,
	COUNT(CASE WHEN availability_60 > 60 THEN 1 END) AS over_60_values,
	ROUND(AVG(availability_60), 2) AS avg_availability_60,
	MEDIAN(availability_60) AS median_availability_60,
	MIN(availability_60) AS min_availability_60, 
	MAX(availability_60) AS max_availability_60
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,over_60_values,avg_availability_60,median_availability_60,min_availability_60,max_availability_60
0,4939,4939,0,1033,0,24.13,23.0,0,60


In [39]:
/* Distribution for availability_60 */ 

WITH listings AS (
	SELECT 
		availability_60,
		COUNT(*) AS listings
	FROM 'detailed_listings.csv'
	GROUP BY availability_60
	ORDER BY listings DESC
),

total_listings AS (
	SELECT 
		COUNT(*) AS total
	FROM 'detailed_listings.csv'
)

SELECT 
	l.availability_60, l.listings,
	ROUND(l.listings/t.total * 100, 2) AS percent_of_total
FROM listings AS l
CROSS JOIN total_listings AS t
ORDER BY percent_of_total DESC;

,availability_60,listings,percent_of_total
0,0,1033,20.92
1,60,216,4.37
2,28,160,3.24
3,29,134,2.71
4,59,123,2.49
...,...,...,...
56,51,37,0.75
57,50,37,0.75
58,54,35,0.71
59,47,33,0.67


In [29]:
/* Checks for availability_90 */ 

SELECT
	COUNT(*) AS total_rows,
	COUNT(availability_90) AS non_null_rows,
	COUNT(*) - COUNT(availability_90) AS null_rows,
	COUNT(CASE WHEN availability_90 = 0 THEN 1 END) AS zero_values,
	COUNT(CASE WHEN availability_90 > 90 THEN 1 END) AS over_90_values,
	ROUND(AVG(availability_90), 2) AS avg_availability_90,
	MEDIAN(availability_90) AS median_availability_90,
	MIN(availability_90) AS min_availability_90, 
	MAX(availability_90) AS max_availability_90
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,over_90_values,avg_availability_90,median_availability_90,min_availability_90,max_availability_90
0,4939,4939,0,798,0,42.57,44.0,0,90


In [40]:
/* Distribution for availability_90 */ 

WITH listings AS (
	SELECT 
		availability_90,
		COUNT(*) AS listings
	FROM 'detailed_listings.csv'
	GROUP BY availability_90
	ORDER BY listings DESC
),

total_listings AS (
	SELECT 
		COUNT(*) AS total
	FROM 'detailed_listings.csv'
)

SELECT 
	l.availability_90, l.listings,
	ROUND(l.listings/t.total * 100, 2) AS percent_of_total
FROM listings AS l
CROSS JOIN total_listings AS t
ORDER BY percent_of_total DESC;

,availability_90,listings,percent_of_total
0,0,798,16.16
1,90,215,4.35
2,58,130,2.63
3,89,118,2.39
4,59,116,2.35
...,...,...,...
86,7,24,0.49
87,19,17,0.34
88,11,17,0.34
89,12,17,0.34


In [31]:
/* Checks for availability_365 */ 

SELECT
	COUNT(*) AS total_rows,
	COUNT(availability_365) AS non_null_rows,
	COUNT(*) - COUNT(availability_365) AS null_rows,
	COUNT(CASE WHEN availability_365 = 0 THEN 1 END) AS zero_values,
	COUNT(CASE WHEN availability_365 > 365 THEN 1 END) AS over_365_values,
	ROUND(AVG(availability_365), 2) AS avg_availability_365,
	MEDIAN(availability_365) AS median_availability_365,
	MIN(availability_365) AS min_availability_365, 
	MAX(availability_365) AS max_availability_365
FROM 'detailed_listings.csv';

,total_rows,non_null_rows,null_rows,zero_values,over_365_values,avg_availability_365,median_availability_365,min_availability_365,max_availability_365
0,4939,4939,0,380,0,206.18,235.0,0,365


In [41]:
/* Distribution for availability_365 */ 

WITH listings AS (
	SELECT 
		availability_365,
		COUNT(*) AS listings
	FROM 'detailed_listings.csv'
	GROUP BY availability_365
	ORDER BY listings DESC
),

total_listings AS (
	SELECT 
		COUNT(*) AS total
	FROM 'detailed_listings.csv'
)

SELECT 
	l.availability_365, l.listings,
	ROUND(l.listings/t.total * 100, 2) AS percent_of_total
FROM listings AS l
CROSS JOIN total_listings AS t
ORDER BY percent_of_total DESC;

,availability_365,listings,percent_of_total
0,0,380,7.69
1,365,128,2.59
2,364,62,1.26
3,334,61,1.24
4,333,56,1.13
...,...,...,...
361,167,2,0.04
362,115,2,0.04
363,100,2,0.04
364,111,2,0.04


In [35]:
SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	availability_30, 
	availability_60,
	availability_90,
	availability_365,
	minimum_nights,
	maximum_nights
FROM 'detailed_listings.csv'
WHERE 
	availability_30 = 0
	OR availability_60 = 0
	OR availability_90 = 0
	OR availability_365 = 0;

,name,description,price,neighbourhood_cleansed,property_type,room_type,availability_30,availability_60,availability_90,availability_365,minimum_nights,maximum_nights
0,Denver’s Peaceful Oasis for Travel Nurses & No...,Enjoy the famous Colorado weather and unplug i...,$106.35,Highland,Entire guesthouse,Entire home/apt,0,27,57,332,29,364
1,Highland Park Guest House,HIGHLAND PARK GUEST HOUSE,$94.75,West Highland,Entire guest suite,Entire home/apt,0,0,12,258,4,1125
2,Midland Lofts 209: In the Heart of Downtown,This condo is right in the heart of Downtown D...,$101.66,CBD,Entire condo,Entire home/apt,0,17,47,322,30,1125
3,1890 townhome in popular LoHi! 2018-BFN-0009355,Perfect space/location for longer term visitor...,$129.61,Highland,Entire townhouse,Entire home/apt,0,30,60,150,29,180
4,Beautiful Home in Charming Location,"Clean, bright and open floor plan! 3 bedroom ...",$292.75,Stapleton,Entire home,Entire home/apt,0,6,16,247,4,1125
...,...,...,...,...,...,...,...,...,...,...,...,...
1729,Platt Park Place | Walk to S Pearl St | 30+ days,Nestled in the highly sought-after neighborhoo...,$127.69,Platt Park,Entire rental unit,Entire home/apt,0,0,27,117,30,365
1730,Green Valley Ranch/DIA stylish & cozy bedroom 2,Take it easy at this unique and tranquil getaw...,$35.03,Gateway - Green Valley Ranch,Private room in home,Private room,0,29,59,334,30,365
1731,Walkable Location: Pet-Friendly Denver Townhome,Newly Built Home | Private Balcony | 8 Mi to D...,$218.00,University Hills,Entire townhouse,Entire home/apt,0,1,29,304,2,1125
1732,La Casa Vallejo - Beautiful 5 bdr Highlands Home,Welcome to a gorgeous 133-year-old stately hom...,"$1,725.46",Highland,Entire home,Entire home/apt,0,28,58,333,1,365


In [44]:
SELECT  
	name, 
	description,
	price,
	neighbourhood_cleansed,
	property_type,
	room_type,
	availability_30, 
	availability_60,
	availability_90,
	availability_365,
	minimum_nights,
	maximum_nights
FROM 'detailed_listings.csv'
WHERE 
	availability_30 > availability_60
	OR availability_60 > availability_90
	OR availability_90 > availability_365;

,name,description,price,neighbourhood_cleansed,property_type,room_type,availability_30,availability_60,availability_90,availability_365,minimum_nights,maximum_nights


In [43]:
WITH ranges AS (
    SELECT
        availability_30 AS available_days_1_30,
        availability_60 - availability_30 AS available_days_31_60,
        availability_90 - availability_60 AS available_days_61_90,
        availability_365 - availability_90 AS available_days_91_365
    FROM 'detailed_listings.csv'
)

SELECT
    ROUND(AVG(available_days_1_30), 2) AS avg_days_1_30,
    ROUND(AVG(available_days_1_30) / 30.0 * 100, 2) || ' ' || '%' AS availability_rate_1_30,

    ROUND(AVG(available_days_31_60), 2) AS avg_days_31_60,
    ROUND(AVG(available_days_31_60) / 30.0 * 100, 2) || ' ' || '%' AS availability_rate_31_60,

    ROUND(AVG(available_days_61_90), 2) AS avg_days_61_90,
    ROUND(AVG(available_days_61_90) / 30.0 * 100, 2) || ' ' || '%' AS availability_rate_61_90,

    ROUND(AVG(available_days_91_365), 2) AS avg_days_91_365,
    ROUND(AVG(available_days_91_365) / 275.0 * 100, 2) || ' ' || '%' AS availability_rate_91_365
FROM ranges;

,avg_days_1_30,availability_rate_1_30,avg_days_31_60,availability_rate_31_60,avg_days_61_90,availability_rate_61_90,avg_days_91_365,availability_rate_91_365
0,9.04,30.12 %,15.09,50.31 %,18.44,61.47 %,163.61,59.49 %
